In [1]:
# packages import
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random

This file contains three parts:
1. Train the forecast model for temperature 
2. Train the forecast model for PV output based on temperature
3. Train the forecast model for load curve based on temperature 
4. First generate scenarios of temperature, and generate scenarios for PV output and load curve based on the generated scenarios

In [2]:
# data reading
data = pd.read_csv(r"22solar_data15.csv", encoding="gbk")
data["hour"] = np.sin(2 * np.pi * (data["hour"] * 60 + data["min"]) / 1440 )
data

,month,day,hour,min,generation,RH,WS,WD,Temp,Rainfall,Vis,Irradiance,SLP
0,1,1,0.000000,0,0.0,83.284,2.5708,91.938,15.720,0.0,13.144,6.2852,1010.65913
1,1,1,0.065403,15,0.0,84.347,2.4569,88.160,15.732,0.0,12.460,6.1747,1010.49913
2,1,1,0.130526,30,0.0,82.773,2.3496,91.985,15.600,0.0,12.853,6.1748,1010.52913
3,1,1,0.195090,45,0.0,83.902,2.6827,96.310,15.487,0.0,12.873,6.3953,1010.49913
4,1,1,0.258819,0,0.0,82.858,2.4456,103.130,15.440,0.0,12.932,6.1749,1010.41913
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,12,31,-0.321439,45,0.0,65.995,3.8198,351.780,13.949,0.0,15.981,8.1723,1010.15913
35036,12,31,-0.258819,0,0.0,65.290,5.1184,352.010,14.184,0.0,15.981,8.3932,1010.02913
35037,12,31,-0.195090,15,0.0,65.939,3.2331,352.920,14.229,0.0,15.981,8.6138,1010.02913
35038,12,31,-0.130526,30,0.0,66.308,3.1078,356.580,14.453,0.0,15.981,8.6140,1010.00913


In [10]:
# build windows for 4 hours based on the historical 24 hours with 15 min intervals
def build_windows(df, x_features, y_features, ctx = 24*4, H = 2*4):
    """
        X contains the historical 24 hours' feature data and corresponding prediction output
        Y contains the current temperature and next 2 hours
        The time interval is 15 min.
    """
    base_features = y_features + x_features
    arr = df[base_features].values
    X_windows, y_windows = [], []
    for i in range(ctx, len(df)-H):
        curr_X = arr[i-ctx:i, :].reshape(-1)
        X_windows.append(curr_X) # This contains the last 24 hours' historical data
        y_windows.append(arr[i:i+H, 0])
    return np.stack(X_windows), np.stack(y_windows)

# Temperature forecast model

In [11]:
# Data buildup
x_temp_features = ['hour', 'Irradiance', 'Rainfall', 'RH', 'SLP', 'WS', 'WD']
y_temp_feature = ['Temp']
H = 2 * 4
ctx = 24 * 4
X_temp, y_temp = build_windows(data, x_temp_features, y_temp_feature, ctx, H)
print(y_temp.shape)
X_temp

(34936, 8)


array([[ 1.57200000e+01,  0.00000000e+00,  6.28520000e+00, ...,
         1.01009913e+03,  2.27630000e+00,  3.42370000e+02],
       [ 1.57320000e+01,  6.54031292e-02,  6.17470000e+00, ...,
         1.01001913e+03,  1.69490000e+00,  3.50400000e+02],
       [ 1.56000000e+01,  1.30526192e-01,  6.17480000e+00, ...,
         1.00995913e+03,  1.16500000e+00,  1.12100000e+01],
       ...,
       [ 1.27990000e+01, -6.59345815e-01,  9.16690000e+00, ...,
         1.01017913e+03,  4.44670000e+00,  3.48860000e+02],
       [ 1.27310000e+01, -6.08761429e-01,  9.05630000e+00, ...,
         1.01024913e+03,  4.27650000e+00,  3.49440000e+02],
       [ 1.26750000e+01, -5.55570233e-01,  9.27710000e+00, ...,
         1.01022913e+03,  4.02130000e+00,  3.57600000e+02]],
      shape=(34936, 768))

In [12]:
# packages used for forecast model
import os
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb

In [13]:
# 1) Quantile objective (works with DMatrix in booster API)
def quantile_objective(q):
    def _obj(preds, dtrain):
        y = dtrain.get_label()          # labels from DMatrix
        resid = y - preds
        grad = np.where(resid > 0, -q, 1 - q)
        hess = np.ones_like(preds) * 1e-6
        return grad, hess
    return _obj

# 2) Train a single horizon + quantile model using booster API
def train_quantile_horizon_booster(X_train, y_train_h, q, base_params=None, num_boost_round=800):
    """
    X_train: (N, F)
    y_train_h: (N,)
    q: quantile in [0,1]
    Returns a trained Booster
    """
    dtrain = xgb.DMatrix(X_train, label=y_train_h)
    params = {
        # 'objective': quantile_objective(q),
        'objective': 'reg:quantileerror',
        'quantile_alpha': q,
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'tree_method': 'hist',
        'nthread': -1,
        'seed': 42
    }
    if base_params:
        params.update(base_params)

    booster = xgb.train(params, dtrain, num_boost_round=num_boost_round)
    return booster

# 3) Train for all horizons and quantiles, and save models with manifest
def train_and_save_quantile_models(X, Y, horizons=None, quantiles=[0.1, 0.5, 0.9],
                                   train_ratio=0.8, model_dir='models', base_params=None, num_boost_round=800):
    """
    X: (N, F)
    Y: (N, H) where H is number of horizons (e.g., 4)
    horizons: list of horizons to train (1-based, e.g., [1,2,3,4])
    """
    if horizons is None:
        horizons = list(range(1, Y.shape[1] + 1))

    N = X.shape[0]
    split = int(train_ratio * N)

    X_train, X_valid = X[:split], X[split:]
    Y_train, Y_valid = Y[:split], Y[split:]

    boosters = [[None for _ in quantiles] for _ in horizons]

    manifest_entries = []
    os.makedirs(model_dir, exist_ok=True)

    for hi, h in enumerate(horizons):
        y_tr = Y_train[:, hi]
        y_va = Y_valid[:, hi]

        for qi, q in enumerate(quantiles):
            b = train_quantile_horizon_booster(X_train, y_tr, q, base_params=base_params, num_boost_round=num_boost_round)
            boosters[hi][qi] = b

            path = os.path.join(model_dir, f'h{h}_q{int(q*100)}.model')
            b.save_model(path)
            manifest_entries.append({'horizon': h, 'quantile': q, 'path': path})

    manifest = {
        'horizons': horizons,
        'quantiles': quantiles,
        'train_ratio': train_ratio,
        'model_dir': model_dir,
        'models': manifest_entries
    }

    manifest_path = os.path.join(model_dir, 'manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    return boosters, manifest, X_valid, Y_valid

# 4) Load models from manifest
def load_models_from_manifest(model_dir='models'):
    manifest_path = os.path.join(model_dir, 'manifest.json')
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

    horizons = manifest['horizons']
    quantiles = manifest['quantiles']

    boosters = []
    for hi, h in enumerate(horizons):
        row = []
        for qi, q in enumerate(quantiles):
            path = manifest['models'][hi * len(quantiles) + qi]['path']
            b = xgb.Booster()
            b.load_model(path)
            row.append(b)
        boosters.append(row)
    return boosters, manifest

# 5) Inference: predict quantiles for new data
def predict_quantiles_from_boosters(X_new, boosters, quantiles=[0.1, 0.5, 0.9]):
    """
    X_new: shape (1, F) or (K, F)
    boosters: list of horizons x quantiles boosters
    Returns: dict horizon -> {quantile: value}
    """
    dtest = xgb.DMatrix(X_new)
    results = {}
    for hi, horizon_boosters in enumerate(boosters):
        h = hi + 1
        row = {}
        for qi, q in enumerate(quantiles):
            preds = horizon_boosters[qi].predict(dtest)
            row[q] = float(preds.ravel()[0])
        results[h] = row
    return results

# 6) Pretty print helper
def print_quantile_forecasts(preds):
    for h in sorted(preds.keys()):
        inner = preds[h]
        print(f"Horizon t+{h}: " + ", ".join([f"q{int(q*100)}={v:.2f}" for q, v in sorted(inner.items())]))

In [16]:
# Train the forecast model
horizon = [i for i in range(H)]
qs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# temp_models, temp_manifest, X_test, Y_test = train_and_save_quantile_models(X=X_temp, Y=y_temp, horizons=horizon, quantiles=qs, model_dir='15temp_models')
# print("Model training finished")
temp_models_loaded, temp_manifest_loaded = load_models_from_manifest('15temp_models')


In [17]:
X_new = X_temp[0:1, :]
preds = predict_quantiles_from_boosters(X_new, temp_models_loaded, qs)
# Sometimes the quantile regression will produce some crossing quantiles
# We need to write a function to enforce there does not exist 
def enforce_non_crossing(preds, quantiles=[0.1, 0.5, 0.9]):
    # preds: dict horizon -> {quantile: value}
    for h in sorted(preds.keys()):
        # ensure the quantile keys are in ascending order
        vals = [preds[h][q] for q in quantiles]
        # monotone non-decreasing across quantiles
        monotone = []
        cur = vals[0]
        monotone.append(cur)
        for v in vals[1:]:
            cur = max(cur, v)
            monotone.append(cur)
        # write back
        for i, q in enumerate(quantiles):
            preds[h][q] = monotone[i]
    return preds

# for comparision
preds = enforce_non_crossing(preds, qs)
print(y_temp[0])
print(preds[1])
print(preds[2])
print(preds[3])
print(preds[4])
print(preds[5])
print(preds[6])
print(preds[7])


[14.48  14.503 14.324 14.391 14.402 14.492 14.503 14.38 ]
{0.1: 14.312324523925781, 0.2: 14.312324523925781, 0.3: 14.364962577819824, 0.4: 14.446157455444336, 0.5: 14.448789596557617, 0.6: 14.457053184509277, 0.7: 14.474556922912598, 0.8: 14.481010437011719, 0.9: 14.53878116607666}
{0.1: 14.355449676513672, 0.2: 14.355449676513672, 0.3: 14.355449676513672, 0.4: 14.440725326538086, 0.5: 14.48066520690918, 0.6: 14.484657287597656, 0.7: 14.484657287597656, 0.8: 14.55221176147461, 0.9: 14.620990753173828}
{0.1: 14.328200340270996, 0.2: 14.33513355255127, 0.3: 14.33513355255127, 0.4: 14.33513355255127, 0.5: 14.346923828125, 0.6: 14.346923828125, 0.7: 14.414942741394043, 0.8: 14.476503372192383, 0.9: 14.711907386779785}
{0.1: 14.347627639770508, 0.2: 14.347627639770508, 0.3: 14.347627639770508, 0.4: 14.352324485778809, 0.5: 14.388666152954102, 0.6: 14.388666152954102, 0.7: 14.436326026916504, 0.8: 14.545369148254395, 0.9: 14.661441802978516}
{0.1: 14.29257869720459, 0.2: 14.29257869720459, 0

## PV output forecast model

In [18]:
# data pre-processing
mdays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
cum_before = np.insert(np.cumsum(mdays), 0, 0)
months = data['month'].to_numpy(dtype=int)
days = data['day'].to_numpy(dtype=int)
data['doy'] = np.sin((cum_before[months-1] + days) * np.pi * 2 / 365)

x_pv_features = ["hour", "Irradiance", "Rainfall", "RH", "SLP", "Temp", "Vis", "WS", "WD", "doy"]
y_pv_feature = ["generation"]
X_pv, y_pv = build_windows(data, x_pv_features, y_pv_feature, ctx, H)
# add a new column of current temperature tp X_pv
# curr_temps = [data["Temp"].to_numpy()[24+dd:-4+dd].reshape(-1, 1) for dd in range(4)]
curr_temps = [data["Temp"].to_numpy()[ctx + dd: -H + dd].reshape(-1, 1) for dd in range(H)]
X_pv = np.hstack([X_pv] + curr_temps)
print(y_pv.shape)
X_pv

(34936, 8)


array([[ 0.        ,  0.        ,  6.2852    , ..., 14.492     ,
        14.503     , 14.38      ],
       [ 0.        ,  0.06540313,  6.1747    , ..., 14.503     ,
        14.38      , 14.279     ],
       [ 0.        ,  0.13052619,  6.1748    , ..., 14.38      ,
        14.279     , 14.29      ],
       ...,
       [ 0.        , -0.65934582,  9.1669    , ..., 13.77      ,
        13.949     , 14.184     ],
       [ 0.        , -0.60876143,  9.0563    , ..., 13.949     ,
        14.184     , 14.229     ],
       [ 0.        , -0.55557023,  9.2771    , ..., 14.184     ,
        14.229     , 14.453     ]], shape=(34936, 1064))

In [19]:
# Train the forecast model
horizon = [i for i in range(H)]
qs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
pv_models, pv_manifest, X_test, Y_test = train_and_save_quantile_models(X=X_pv, Y=y_pv, horizons=horizon, quantiles=qs, model_dir='15pv_models')
print(f"PV forecast model trained.")
pv_models_loaded, pv_manifest_loaded = load_models_from_manifest('15pv_models')


/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_7611/299027823.py:68: UserWarning: [22:10:54] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_7611/299027823.py:68: UserWarning: [22:11:34] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_7611/299027823.py:68: UserWarning: [22:12:13] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrs

PV forecast model trained.


In [24]:
X_new = X_pv[40:41, :]
preds = predict_quantiles_from_boosters(X_new, pv_models_loaded, qs)
preds = enforce_non_crossing(preds, qs)
print(y_pv[40])
print(preds[1])
print(preds[2])
print(preds[3])
print(preds[4])
print(preds[5])
print(preds[6])
print(preds[7])

[ 9.704   10.69925 11.6945  12.68975 13.685   14.20875 14.7325  15.25625]
{0.1: 8.365449905395508, 0.2: 9.28013801574707, 0.3: 9.585037231445312, 0.4: 9.779064178466797, 0.5: 9.779064178466797, 0.6: 10.152151107788086, 0.7: 10.152151107788086, 0.8: 10.17706298828125, 0.9: 10.17706298828125}
{0.1: 8.3201265335083, 0.2: 10.57420539855957, 0.3: 10.759278297424316, 0.4: 10.767965316772461, 0.5: 11.33827018737793, 0.6: 11.33827018737793, 0.7: 11.33827018737793, 0.8: 11.655200958251953, 0.9: 11.655200958251953}
{0.1: 8.678932189941406, 0.2: 11.200470924377441, 0.3: 11.785730361938477, 0.4: 11.79054069519043, 0.5: 11.79054069519043, 0.6: 12.694398880004883, 0.7: 12.694398880004883, 0.8: 12.928936958312988, 0.9: 12.928936958312988}
{0.1: 8.059590339660645, 0.2: 12.5562744140625, 0.3: 12.987637519836426, 0.4: 13.214622497558594, 0.5: 13.214622497558594, 0.6: 13.214622497558594, 0.7: 13.240839958190918, 0.8: 14.308106422424316, 0.9: 14.308106422424316}
{0.1: 6.752194404602051, 0.2: 13.3859138488

## Load Forecasting Data Processing
1. Read data from the supermarket data
2. DO the simple load forecasting based on temperatures and hours, but still use XGBoost

In [21]:
load_data = pd.read_csv(r"building data/2zonesupermarket15.csv", encoding="gbk")
load_data

,outdoor temperature,load,date,time
0,0.75,9310.5,0.000000,0.000000
1,1.50,9310.5,0.000000,0.065403
2,2.25,9310.5,0.000000,0.130526
3,3.00,9310.5,0.000000,0.195090
4,2.00,9310.5,0.000000,0.258819
...,...,...,...,...
35035,0.00,9310.5,-0.017213,-0.321439
35036,0.00,9310.5,-0.017213,-0.258819
35037,0.00,9310.5,-0.017213,-0.195090
35038,0.00,9310.5,-0.017213,-0.130526


In [25]:
x_load_features = ['outdoor temperature', 'time']
y_load_features = ['load']
X_load, y_load = build_windows(load_data, x_load_features, y_load_features)
curr_temps = [load_data['outdoor temperature'].to_numpy()[ctx+dd:-H+dd].reshape(-1, 1) for dd in range(H)]
X_load = np.hstack([X_load] + curr_temps)
print(y_load.shape)
X_load

(34936, 8)


array([[ 9.31050000e+03,  7.50000000e-01,  0.00000000e+00, ...,
        -3.50000000e+00, -3.75000000e+00, -4.00000000e+00],
       [ 9.31050000e+03,  1.50000000e+00,  6.54031292e-02, ...,
        -3.75000000e+00, -4.00000000e+00, -4.25000000e+00],
       [ 9.31050000e+03,  2.25000000e+00,  1.30526192e-01, ...,
        -4.00000000e+00, -4.25000000e+00, -4.50000000e+00],
       ...,
       [ 1.38120000e+04,  0.00000000e+00, -6.59345815e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.38120000e+04,  5.00000000e-01, -6.08761429e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.38120000e+04,  1.00000000e+00, -5.55570233e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(34936, 296))

In [26]:
horizon = [i for i in range(H)]
qs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
load_models, load_manifest, X_test, y_test = train_and_save_quantile_models(X=X_load, Y=y_load, horizons=horizon, quantiles=qs, model_dir='15load_models')
print(f"Load model trained.")
load_models_loaded, load_manifest_loaded = load_models_from_manifest('15load_models')

/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_7611/299027823.py:68: UserWarning: [09:40:10] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_7611/299027823.py:68: UserWarning: [09:40:13] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_7611/299027823.py:68: UserWarning: [09:40:17] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrs

Load model trained.


In [28]:
X_new = X_load[222:223, :]
preds = predict_quantiles_from_boosters(X_new, load_models_loaded, qs)
preds = enforce_non_crossing(preds, qs)
print(y_load[222])
# print(preds)
print(preds[1])
print(preds[2])
print(preds[3])
print(preds[4])
print(preds[5])
print(preds[6])


[60322.5 60322.5 62130.  62130.  62130.  62130.  62130.  62130. ]
{0.1: 28811.98046875, 0.2: 30388.53515625, 0.3: 60322.49609375, 0.4: 60322.5, 0.5: 60322.5, 0.6: 62130.0, 0.7: 62130.0, 0.8: 62130.0, 0.9: 62130.0}
{0.1: 28811.98046875, 0.2: 30388.53515625, 0.3: 60322.2734375, 0.4: 60322.4765625, 0.5: 60322.62890625, 0.6: 62130.0, 0.7: 62130.0, 0.8: 62130.0, 0.9: 62130.0}
{0.1: 28811.98046875, 0.2: 30388.53515625, 0.3: 62129.9609375, 0.4: 62129.9609375, 0.5: 62129.9609375, 0.6: 62130.0, 0.7: 62130.0, 0.8: 62130.0, 0.9: 62130.0}
{0.1: 28811.98046875, 0.2: 30388.53515625, 0.3: 62129.9140625, 0.4: 62129.9609375, 0.5: 62129.9609375, 0.6: 62130.0, 0.7: 62130.0, 0.8: 62130.0, 0.9: 62130.0}
{0.1: 28811.98046875, 0.2: 30388.53515625, 0.3: 62129.9609375, 0.4: 62129.9609375, 0.5: 62129.9609375, 0.6: 62130.0, 0.7: 62130.0, 0.8: 62130.0, 0.9: 62130.0}
{0.1: 28811.98046875, 0.2: 30388.53515625, 0.3: 62129.91015625, 0.4: 62129.9609375, 0.5: 62129.9609375, 0.6: 62130.0, 0.7: 62130.0, 0.8: 62130.0, 0.9